### 生成EnrichmentScore.csv (排除全0),并生成用于t-test的csv文件（向量格式）

In [1]:
import pandas as pd
import glob
import os

TCN_num =10 

CellTypeVec_path = '../../data/EnrichScoreMatrix/CellTypeVec_List.csv'
df = pd.read_csv(CellTypeVec_path)
CellType_list = df.iloc[:, 0].tolist()
print(CellType_list)


input_folder = '../../data/EnrichScoreMatrix/'
output_folder = input_folder + 'Score'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

for i in range(0, TCN_num):
    tcn_data = {
        'Condition': [],
        'EnrichSco': [],
        'CellType': []
    }
    # 对于细胞类型j
    for j, cell_type in enumerate(CellType_list):
        # 遍历所有样本
        for file in glob.glob(os.path.join(input_folder, '*_EnrichScoreMatrix.csv')):
            data = pd.read_csv(file,header=None)

            # 从文件名中提取condition
            file_name = os.path.basename(file)
            condition = file_name.split('_')[0]  
            # 第i个TCN中，细胞类型j的富集得分
            enrichment_sco = data.iloc[j, i]  

            tcn_data['Condition'].append(condition)
            tcn_data['EnrichSco'].append(enrichment_sco)
            tcn_data['CellType'].append(cell_type)

        tcn_df = pd.DataFrame(tcn_data)

        # 保证同一celltype的5行EnrichSco不全为0
        tcn_df = tcn_df.groupby('CellType').filter(lambda x: (x['EnrichSco'] != 0).sum() > 0)

        # 保存结果
        output_file = os.path.join(output_folder, f'TCN{i+1}_EnrichmentScore.csv')
        tcn_df.to_csv(output_file, index=False)

for tcn in range(1, TCN_num+1):

    input_file = os.path.join(input_folder, 'Score', f'TCN{tcn}_EnrichmentScore.csv')
    output =input_folder+'/t_test/bias_result'
    if not os.path.exists(output):
        os.makedirs(output)
    output_file = os.path.join(input_folder, 't_test', f'TCN{tcn}.csv')
    
    data = pd.read_csv(input_file)

    grouped = data.groupby('CellType')
    result = []

    for cell_type, group in grouped:
        vectors = {}
        for condition in group['Condition'].unique():
            enrich_sco_vector = tuple(float(v) for v in group[group['Condition'] == condition]['EnrichSco'].values)
            vectors[condition] = enrich_sco_vector
        for condition, vector in vectors.items():
            result.append({'CellType': cell_type, 'Condition': condition, 'EnrichSco': vector})

    result_df = pd.DataFrame(result)

    # 去掉 -0.0
    if 'EnrichSco' in result_df.columns:
        def replace_negative_zero(value):
            if isinstance(value, tuple):
                return tuple(0 if v == -0.0 else v for v in value)
            return value
        result_df['EnrichSco'] = result_df['EnrichSco'].apply(replace_negative_zero)
    else:
        print(f"[WARNING] 'EnrichSco' column not found in result_df for TCN{tcn}.")

    result_df.to_csv(output_file, index=False)
    print(f"[SAVE] {output_file}")


['B-Lym', 'B-Nai', 'GC B-DZ', 'MAST', 'Macrophage', 'Monocyte', 'NK', 'T-Cyt', 'T-Nai', 'T-Reg', 'Tumor', 'pDC']
[SAVE] ../../data/EnrichScoreMatrix/t_test\TCN1.csv
[SAVE] ../../data/EnrichScoreMatrix/t_test\TCN2.csv
[SAVE] ../../data/EnrichScoreMatrix/t_test\TCN3.csv
[SAVE] ../../data/EnrichScoreMatrix/t_test\TCN4.csv
[SAVE] ../../data/EnrichScoreMatrix/t_test\TCN5.csv
[SAVE] ../../data/EnrichScoreMatrix/t_test\TCN6.csv
[SAVE] ../../data/EnrichScoreMatrix/t_test\TCN7.csv
[SAVE] ../../data/EnrichScoreMatrix/t_test\TCN8.csv
[SAVE] ../../data/EnrichScoreMatrix/t_test\TCN9.csv
[SAVE] ../../data/EnrichScoreMatrix/t_test\TCN10.csv


### 筛选p-value<0.05的细胞类型(需要先进行T检验)

In [2]:
from pathlib import Path
import pandas as pd

source_folder = Path("../../data/EnrichScoreMatrix/t_test/bias_result")
final_folder = source_folder / "final"
final_folder.mkdir(parents=True, exist_ok=True)

for csv_path in source_folder.rglob("*.csv"):
    # 跳过 final 下的文件
    if final_folder in csv_path.parents:
        continue

    df = pd.read_csv(csv_path)
    col_map = {c: c.strip() for c in df.columns}
    df.rename(columns=col_map, inplace=True)

    if "p_value_left" in df.columns and "p_value_right" in df.columns:
        out_path = final_folder / csv_path.name
        # 过滤
        filtered_df = df[(df["p_value_left"] < 0.05) | (df["p_value_right"] < 0.05)].copy()
        if filtered_df.empty:
            print(f"[skip empty] {csv_path}")
            continue

        filtered_df.to_csv(out_path, index=False)
        print(f"[wrote] {out_path}")

    else:
        continue

for csv_path in final_folder.glob("*.csv"):
    df = pd.read_csv(csv_path)

    if {"p_value_left", "p_value_right"} <= set(df.columns):
        df["p_value"] = df[["p_value_left", "p_value_right"]].min(axis=1)

        if "CellType" not in df.columns:
            print(f"[skip no CellType] {csv_path}")
            continue

        out_df = df[["CellType", "p_value"]].copy()

        if out_df.empty:
            print(f"[skip empty] {csv_path}")
            continue

        out_df.to_csv(csv_path, index=False)
        print(f"[wrote minimal] {csv_path}")

    else:
        continue

print("Done: filtered results and wrote minimal p_value files into:", final_folder.as_posix())


[wrote] ..\..\data\EnrichScoreMatrix\t_test\bias_result\final\TCN1.csv
[skip empty] ..\..\data\EnrichScoreMatrix\t_test\bias_result\TCN10.csv
[skip empty] ..\..\data\EnrichScoreMatrix\t_test\bias_result\TCN2.csv
[wrote] ..\..\data\EnrichScoreMatrix\t_test\bias_result\final\TCN3.csv
[wrote] ..\..\data\EnrichScoreMatrix\t_test\bias_result\final\TCN4.csv
[wrote] ..\..\data\EnrichScoreMatrix\t_test\bias_result\final\TCN5.csv
[skip empty] ..\..\data\EnrichScoreMatrix\t_test\bias_result\TCN6.csv
[wrote] ..\..\data\EnrichScoreMatrix\t_test\bias_result\final\TCN7.csv
[skip empty] ..\..\data\EnrichScoreMatrix\t_test\bias_result\TCN8.csv
[skip empty] ..\..\data\EnrichScoreMatrix\t_test\bias_result\TCN9.csv
[wrote minimal] ..\..\data\EnrichScoreMatrix\t_test\bias_result\final\TCN1.csv
[wrote minimal] ..\..\data\EnrichScoreMatrix\t_test\bias_result\final\TCN3.csv
[wrote minimal] ..\..\data\EnrichScoreMatrix\t_test\bias_result\final\TCN4.csv
[wrote minimal] ..\..\data\EnrichScoreMatrix\t_test\bias_